In [1]:
import os
from dotenv import load_dotenv,find_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import AzureChatOpenAI,AzureOpenAIEmbeddings
from langgraph.graph import StateGraph,MessagesState
from langgraph.prebuilt import  ToolNode,tools_condition
from langchain_core.tools import tool
from IPython.display import Image
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(find_dotenv(),override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version="2025-01-01-preview"


In [2]:
@tool
def get_weather(location:str)->str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

tools=[get_weather]

llm = AzureChatOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version=api_version
    ).bind_tools(tools)

In [6]:
def chatbot(state:MessagesState):
    """it will also call tools node"""
    res=llm.invoke(state["messages"])
    return {"messages":res}

builder=StateGraph(MessagesState)
builder.add_node("chatbot",chatbot) 
tool_node=ToolNode(tools)
builder.add_node("tools",tool_node)

builder.set_entry_point("chatbot")
builder.add_conditional_edges(
    "chatbot",
    tools_condition)
builder.add_edge("tools","chatbot")

checkpoint=MemorySaver()
graph=builder.compile( checkpointer=checkpoint, interrupt_before=["tools"])
# display(Image(graph.get_graph().draw_mermaid_png()))

config={"configurable":{"thread_id":"1"}}
intput_message=HumanMessage(content="What's the weather like in india?")
res=graph.invoke({"messages":[intput_message]},config=config)
res
#res["messages"][-1].content

{'messages': [HumanMessage(content="What's the weather like in india?", additional_kwargs={}, response_metadata={}, id='cbeffee3-3235-45ca-ab99-f8e2c2d8c45f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_HFxlf9mnVupKY1vRwgHb1lvE', 'function': {'arguments': '{"location":"India"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_63d8723934', 'id': 'chatcmpl-DMpaKFjm8bF1dR3uWQf4yAivw4baz', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='run--2427366e-fa0d-408d-ac3c-9cc5a93b2407-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'India'}, '

In [7]:
# snapshot=graph.get_state(config)
# snapshot.next
graph.invoke(None,config=config)


{'messages': [HumanMessage(content="What's the weather like in india?", additional_kwargs={}, response_metadata={}, id='cbeffee3-3235-45ca-ab99-f8e2c2d8c45f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_HFxlf9mnVupKY1vRwgHb1lvE', 'function': {'arguments': '{"location":"India"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_63d8723934', 'id': 'chatcmpl-DMpaKFjm8bF1dR3uWQf4yAivw4baz', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='run--2427366e-fa0d-408d-ac3c-9cc5a93b2407-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'India'}, '